## tl;dr

This notebook inventories every benchmark JSON file under `results/`, normalizes compatible raw cases, and emits descriptive metrics plus a machine-readable report-builder input. The executed summary below is authoritative for the current files. No model winner or frontier-parity conclusion is declared.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

working_directory = Path.cwd().resolve()
repository_root = working_directory if (working_directory / 'results').is_dir() else working_directory.parent
if not (repository_root / 'results').is_dir():
    raise FileNotFoundError('Run from the repository root or notebooks/ directory.')
sys.path.insert(0, str(repository_root))

from analysis.benchmark_results import build_analysis, format_table, write_analysis_artifact

results_root = repository_root / 'results'
artifact_path = repository_root / 'reports' / 'benchmark-analysis-input.json'
analysis = build_analysis(results_root)
quality = analysis['dataQuality']

print(f"Loaded {quality['inputFileCount']} JSON files: {quality['rawReportCount']} raw reports and {quality['fileKinds'].get('derived', 0)} derived files.")
print(f"Raw completion: {quality['completeRawReportCount']} complete, {quality['partialRawReportCount']} partial.")
print(f"Normalized cases: {quality['caseObservations']} observed, {quality['completedCaseObservations']} completed, {quality['failedCaseObservations']} failed.")
print(f"Present suites: {', '.join(quality['presentBenchmarkKinds']) or 'none'}; missing suites: {', '.join(quality['missingBenchmarkKinds']) or 'none'}.")
print('Conclusion status: provisional measurements only; frontier parity is not established.')

Loaded 27 JSON files: 17 raw reports and 1 derived files.
Raw completion: 14 complete, 3 partial.
Normalized cases: 354 observed, 330 completed, 24 failed.
Present suites: code, loop, text, vision; missing suites: none.
Conclusion status: provisional measurements only; frontier parity is not established.


## Context & Methods

The reader is a future benchmark author, analyst, or report builder. Inputs are discovered recursively from `results/**/*.json`; no filename allowlist is needed. Raw text, code, vision, and loop reports are normalized by schema markers. Derived summaries are inventoried for provenance but excluded from measurement aggregation to prevent double counting. Unknown or malformed files remain visible as data-quality warnings.

### Key Assumptions

- A completed model case has `status == "completed"`; answer quality remains represented by its numeric score.
- Failure rate measures operationally incomplete cases, while zero-score rate measures completed or attempted cases that earned no credit.
- Partial runs contribute only observed cases. Expected, observed, and missing case counts remain separate; missing cases are not silently scored as model failures.
- Latency and throughput summarize available response telemetry only. Missing telemetry is retained as null.
- Aggregates combine recorded case observations and do not correct for repeated tasks, unequal run counts, quantization differences, or hardware-state differences.
- Loop deltas compare self-refinement and cross-specialist treatments with the direct treatment within the same loop artifact.

In [2]:
method_receipt = {
    'inputPattern': 'results/**/*.json',
    'rawKinds': ['text', 'code', 'vision', 'loop'],
    'derivedFilesCountedAsMeasurements': False,
    'partialRunPolicy': 'include observed cases; report expected/observed/missing separately',
    'rankingProduced': False,
    'frontierComparisonPerformed': False,
}
print(json.dumps(method_receipt, indent=2))

{
  "inputPattern": "results/**/*.json",
  "rawKinds": [
    "text",
    "code",
    "vision",
    "loop"
  ],
  "derivedFilesCountedAsMeasurements": false,
  "partialRunPolicy": "include observed cases; report expected/observed/missing separately",
  "rankingProduced": false,
  "frontierComparisonPerformed": false
}


## Data

The inventory preserves path, byte size, SHA-256, schema classification, completion state, and parse status for every JSON input. Tables are bounded for readability; the generated artifact retains every row.

In [3]:
inventory_columns = ['path', 'kind', 'sizeBytes', 'runComplete', 'changedDuringRead', 'includedInMeasurements', 'parseError']
print(format_table(analysis['tables']['reportInventory'], inventory_columns, limit=30))

path                                             | kind    | sizeBytes | runComplete | changedDuringRead | includedInMeasurements | parseError
-------------------------------------------------+---------+-----------+-------------+-------------------+------------------------+-----------
results/code-finalists-isolated-r2.json          | code    | 373340    | True        | False             | True                   | None      
results/derived-stock/summary.json               | derived | 52804     | False       | False             | False                  | None      
results/dual-resident-e4b.json                   | text    | 41527     | True        | False             | True                   | None      
results/dual-resident-granite.json               | text    | 31687     | True        | False             | True                   | None      
results/llamacpp-placement-devstral-b10566-r2.js | unknown | 19845     | False       | False             | False                  | None      

In [4]:
coverage_columns = [
    'source', 'benchmarkKind', 'model', 'modelStatus', 'runComplete',
    'expectedCases', 'observedCases', 'completedCases', 'failedCases', 'missingCases', 'coverageRate',
]
print(format_table(analysis['tables']['modelCoverage'], coverage_columns, limit=40))

source                                        | benchmarkKind | model                                            | modelStatus | runComplete | expectedCases | observedCases | completedCases | failedCases | missingCases | coverageRate
----------------------------------------------+---------------+--------------------------------------------------+-------------+-------------+---------------+---------------+----------------+-------------+--------------+-------------
results/code-finalists-isolated-r2.json       | code          | mistralai_devstral-small-2-24b-instruct-2512     | completed   | True        | 10            | 10            | 10             | 0           | 0            | 1.0         
results/code-finalists-isolated-r2.json       | code          | granite-4.1-8b                                   | completed   | True        | 10            | 10            | 10             | 0           | 0            | 1.0         
results/code-finalists-isolated-r2.json       | code          | 

## Results

These are descriptive measurements sorted by model and category identity, not a leaderboard. `meanScoreAllAttempts` uses recorded case slots; `meanScoreCompleted` excludes operationally failed cases. Coverage fields show where partial runs limit interpretation.

In [5]:
model_columns = [
    'model', 'observations', 'completedCases', 'failedCases', 'failureRate',
    'meanScoreAllAttempts', 'meanScoreCompleted', 'zeroScoreRate',
    'meanLatencyMs', 'p95LatencyMs', 'meanTokensPerSecond',
    'expectedCases', 'missingCases', 'coverageRate', 'partialRunObservations',
]
print(format_table(analysis['tables']['perModel'], model_columns, limit=50))

model                                            | observations | completedCases | failedCases | failureRate | meanScoreAllAttempts | meanScoreCompleted | zeroScoreRate | meanLatencyMs | p95LatencyMs | meanTokensPerSecond | expectedCases | missingCases | coverageRate | partialRunObservations
-------------------------------------------------+--------------+----------------+-------------+-------------+----------------------+--------------------+---------------+---------------+--------------+---------------------+---------------+--------------+--------------+-----------------------
deepseek/deepseek-r1-0528-qwen3-8b               | 20           | 20             | 0           | 0.0         | 0.098571             | 0.098571           | 0.75          | 17235.65      | 48961.85     | 29.747              | 20            | 0            | 1.0          | 0                     
google/gemma-4-12b                               | 22           | 20             | 2           | 0.090909    | 0.318182  

In [6]:
category_columns = [
    'model', 'benchmarkKind', 'category', 'observations', 'completedCases',
    'failureRate', 'meanScoreAllAttempts', 'meanScoreCompleted',
    'meanLatencyMs', 'meanTokensPerSecond',
]
print(format_table(analysis['tables']['perModelCategory'], category_columns, limit=40))

model                              | benchmarkKind | category         | observations | completedCases | failureRate | meanScoreAllAttempts | meanScoreCompleted | meanLatencyMs | meanTokensPerSecond
-----------------------------------+---------------+------------------+--------------+----------------+-------------+----------------------+--------------------+---------------+--------------------
deepseek/deepseek-r1-0528-qwen3-8b | text          | coding           | 2            | 2              | 0.0         | 0.0                  | 0.0                | 6529.0        | 31.427             
deepseek/deepseek-r1-0528-qwen3-8b | text          | critique         | 2            | 2              | 0.0         | 0.0                  | 0.0                | 20979.5       | 31.236             
deepseek/deepseek-r1-0528-qwen3-8b | text          | extraction       | 2            | 2              | 0.0         | 0.0                  | 0.0                | 18054.5       | 30.719             
deepseek/d

In [7]:
loop_method_columns = [
    'source', 'method', 'runComplete', 'caseSlots', 'completedCases', 'coverageRate',
    'passRateAllSlots', 'calls', 'wallSeconds', 'completionTokens',
]
loop_delta_columns = [
    'source', 'method', 'runComplete', 'passRateDeltaVsDirect', 'scoreDeltaVsDirect',
    'callsDeltaVsDirect', 'wallSecondsDeltaVsDirect', 'completionTokensDeltaVsDirect',
]
if analysis['tables']['loopMethods']:
    print('Loop method measurements')
    print(format_table(analysis['tables']['loopMethods'], loop_method_columns, limit=30))
    print('\nLoop deltas versus direct')
    print(format_table(analysis['tables']['loopDeltas'], loop_delta_columns, limit=30))
else:
    print('No raw loop-ablation JSON is present. Loop deltas are not measured yet.')

Loop method measurements
source                                           | method          | runComplete | caseSlots | completedCases | coverageRate | passRateAllSlots | calls | wallSeconds | completionTokens
-------------------------------------------------+-----------------+-------------+-----------+----------------+--------------+------------------+-------+-------------+-----------------
results/loop-ablation-e4b-granite-dual-resident- | crossSpecialist | True        | 12        | 12             | 1.0          | 0.586905         | 36    | 250.983     | 5042            
results/loop-ablation-e4b-granite-dual-resident- | direct          | True        | 12        | 12             | 1.0          | 0.592857         | 12    | 165.796     | 1895            
results/loop-ablation-e4b-granite-dual-resident- | selfRefine      | True        | 12        | 12             | 1.0          | 0.192063         | 36    | 978.237     | 9335            
results/loop-ablation-e4b-granite-r2-clean.json  |

In [8]:
write_analysis_artifact(analysis, artifact_path)
artifact_sha256 = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
print(f'Wrote {artifact_path.relative_to(repository_root).as_posix()}')
print(f'Bytes: {artifact_path.stat().st_size}; SHA-256: {artifact_sha256}')

Wrote reports/benchmark-analysis-input.json
Bytes: 530783; SHA-256: eedb28b8f7c7be2684998a04359c01f7a48e30c08ba596010668807f008f4823


## Takeaways

The final status is generated from data-quality fields rather than inferred from whichever model has the largest observed score. This keeps incomplete measurements and missing suites visible to downstream report builders.

In [9]:
if quality['partialRawReportPaths']:
    print('Partial raw reports retained:')
    for path in quality['partialRawReportPaths']:
        print(f'  - {path}')
else:
    print('No partial raw reports were detected.')
if quality['missingBenchmarkKinds']:
    print(f"New measurements are still required for: {', '.join(quality['missingBenchmarkKinds'])}.")
if not analysis['tables']['loopDeltas']:
    print('No specialist-loop improvement or regression is established because no loop delta is present.')
print('No final model winner is declared. Frontier parity remains not established.')

Partial raw reports retained:
  - results/screen-all-installed.json
  - results/screen-qwen3.6-27b-thirdparty-isolated-r1.json
  - results/screen-qwen3.6-27b-thirdparty-isolated-r2-clean.json
No final model winner is declared. Frontier parity remains not established.
